In [ ]:
import numpy as np
from scipy.constants import k, h, pi, N_A,R
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import pandas as pd
R_con=R

In [ ]:
#calculate entropy
def calculate_entropy(solvent_property,solute_property):
    """
    The function is based on this paper(https://pubs.acs.org/doi/10.1021/acs.jctc.9b00214)
    For our system, we will only consider the translation entropy and rotation entropy since we consider that
    the external addition and internal linking do not differ significantly in terms of vibration entropy. 
    As for the cavity entropy, the situation is more complicated and enthaply should be involved accounting
    for "entropy-enthalpy compensation" to matche the experimental results requiring extra work deviates from 
    our simulation objectives.So we assume that the term of external addition has little difference with the 
    one of internal linking.
    
    Args:
    Solvent_property:A list to describe the property of solvent including 
    [the molecular weight,the volume of a solvent molecule,the mass density,temperature]
    i.e.[M_S,V_S,rho,T];
    Solute_property:A list to describe the property of solute including
    [the molecular weight,the volume of a solute molecule,radius of gyration,the rotational symmetry number,the average moment of inertia]
    i.e.[m,V_M,,r_g,sigma,I];

    Return: Translational entropy, rotational entropy and their summation
 
    """
    M_S,V_S,rho,T=solvent_property
    m,V_M,r_g,sigma,I=solute_property

    """Calculate translation entropy (St:R/mol)"""
    # Calculate the volume of free space per solvent particle(unit:A^3) 
    V_free=(M_S/(N_A*rho))*1e24-V_S
    # Calculate the volume of the cavity
    V_c=(V_M**(1/3)+V_free**(1/3))**3
    # Calculate the radius of the cavity
    r_c=(3*V_c/(4*pi))**(1/3)

    # Calculate the probility per solvent particle of "hopping" to an adjacent cavity
    x=max(0,V_free**(2/3)-V_M**(2/3))/(V_free**(2/3)+V_S**(2/3))
    N_x=4*((4*pi/3)**(2/3))*((r_c**2)/(V_free**(2/3)+V_S**(2/3)))
    # Calculate the average number of accessible cavities
    N_c=1+N_x*(1/(1-x)-1)

    # Calculate the volume of the box 
    V=V_c*N_c*1e-30
    m=m*1e-3/N_A
    # Calculate the translational partition function
    q_t = (((2 * pi *m* k * T) / (h**2)) ** (3/2))*V
    # Calculate the translational entropy(the entropy penalty of bringing a gas at standard conditions to a 1 M concentration is included)
    S_t = np.log(q_t) + 2.5-6.4*4.184/R_con

    """Calculate rotational entropy(Sr:R/mol)"""
    # There are three condtions to be discussed
    if (r_c>=r_g) and (r_c-r_g>=((3 / (4 * pi)) ** (2/3)) * (V_free ** (1/3))):
        # Calculate the rotational partition function
        q_r = (pi**0.5 / sigma) * (8 * pi**2 * I * k * T / h**2) ** (3/2)
        # Calculate the translational entropy loss due to rotation
        Sr_loss=3*np.log((r_c-r_g)/r_c)
        # Calculate the rotational entropy
        S_r = np.log(q_r) +1.5+Sr_loss
    elif (r_c>=r_g) and (r_c-r_g<((3 / (4 * pi)) ** (2/3)) * (V_free ** (1/3))):
        q_r = (pi**0.5 / sigma) * (8 * pi**2 * I * k * T / h**2) ** (3/2)
        Sr_loss=3*np.log((3 / (4 * pi)) ** (2/3) * V_free ** (1/3)/r_c)
        S_r=np.log(q_r)+1.5+Sr_loss 
    else:
        """Calculate angle θ_0"""
        # Calculate the radius of the free space per solvent particle(unit:A)
        r_free = (3 * V_free / (4 * pi)) ** (1/3)
        theta_0 = 2 * np.arccos(r_g / np.sqrt(r_g**2 + r_free**2))
        q_r = (8 * pi * I * k * T / h**2) ** (3/2)*theta_0**2
        Sr_loss=3*np.log((3 / (4 * pi)) ** (2/3) * V_free ** (1/3)/r_c)
        S_r=np.log(q_r)+1.5+Sr_loss
    
    return S_t,S_r,S_t+S_r

In [ ]:
# Case1：solvent-water;solute-Water
solvent_property=[18,17.06537214,1,298.15]
solute_properpy=[18,17.06537214,0.683,2,1.839e-47]
S_t,S_r,S=calculate_entropy(solvent_property,solute_properpy)
print(f"Translational Entropy: {S_t} R\nRotational Entropy: {S_r} R\nEntropy: {S} R")

In [ ]:
# Case2: solvent-water;soluet-acetic acid
solvent_property=[18,17.06537214,1,298.15]
solute_properpy=[60,43.47396368,1.585,1,1.017e-45]
S_t,S_r,S=calculate_entropy(solvent_property,solute_properpy)
print(f"Translational Entropy: {S_t} R\nRotational Entropy: {S_r} R\nEntropy: {S} R")

In [ ]:
# Case3: solvent-water;solute-acetone
solvent_property=[18,17.06537214,1,298.15]
solute_properpy=[58.042,49.31160768,1.731,2,1.125e-45]
S_t,S_r,S=calculate_entropy(solvent_property,solute_properpy)
print(f"Translational Entropy: {S_t} R\nRotational Entropy: {S_r} R\nEntropy: {S} R")

In [ ]:
# Case4: our system
solvent_property=[49.8505,38.91547731,1.024396285,393]
solute_property={"SBU":[1731.072,291.015624,3.547178451,12,1.55E-43],
"BTB":[438.435,233.2037763,5.376506997,6,1.41E-43],
"MOL_1":[2517.908,682.4617454,8.997496131,1,1.32E-42],
"MOL_1'":[3810.545,744.0893657,9.852656346,1,2.91E-42],
"MOL_2":[4203.963,943.2990032,9.609332442,1,3.60E-42],
"MOL_3":[4158.946,905.8820967,9.509379832,1,3.45E-42],
"assembly_2024-09-09_22-52":[92725.542,12868.74891,45.04716396,1,1.86E-39],
"assembly_2024-09-09_22-20":[66260.452,8784.479443,34.18442534,1,7.84E-40],
"assembly_2024-09-10_18-51":[279444.233,37125.02875,68.01290084,1,1.36E-38],
"assembly_2024-09-10_11-58":[276860.975,37048.07589,67.59314617,1,1.32E-38],
"assembly_2024-09-06_21-37":[95308.8,12941.57887,36.52014625,1,1.40E-39],
"assembly_2024-09-06_21-45":[50574.459,7224.319179,36.75270502,1,7.12E-40],
"assembly_2024-09-28_01-35":[287574.549,36685.33551,58.08685289,1,1.05E-38],
"assembly_2024-09-11_04-26":[539587.736,69384.25858,72.11619521,1,3.01E-38],
"assembly_2024-09-10_13-41":[275569.346,35812.81048,61.24718565,1,1.11E-38],
"assembly_2024-09-27_19-59":[239237.482,32082.19592,54.75337105,1,7.90E-39],
"assembly_2024-09-09_20-42":[11127.556,1651.21052,16.63425179,1,3.14E-41],
"assembly_2024-09-06_23-13":[91433.913,12463.52588,39.99187599,1,1.64E-39],
"assembly_2024-09-27_22-20":[50021.249,6683.289346,27.83672162,1,4.42E-40],
"assembly_2024-09-27_22-20":[30351.988,3910.862706,26.32585669,1,2.14E-40]}
# Create a blank DataFrame to save the result
results = pd.DataFrame(columns=["Translational Entropy", "Rotational Entropy", "Entropy"])
for key, value in solute_property.items():
    S_t,S_r,S=calculate_entropy(solvent_property,value)
    results.loc[key] = [S_t, S_r, S]
    print(f"{key}:")
    print(f"Translational Entropy: {S_t} R\nRotational Entropy: {S_r} R\nEntropy: {S} R")
    print()
# Save the result to the csv
results.to_csv("entropy_results.csv",index=True)

In [ ]:
#Visualize the result
SBU=[5,13,23,38,99,118,225]
BTB=[6,19,27,62,164,182,363]
solvent_property=[49.8505,38.91547731,1.024396285,393]
solute_property={
"MOL_5":[11127.556,1651.21052,16.63425179,1,3.14E-41],
"MOL_13":[30351.988,3910.862706,26.32585669,1,2.14E-40],
"MOL_23":[50574.459,7224.319179,36.75270502,1,7.12E-40],
"MOL_38":[91433.913,12463.52588,39.99187599,1,1.64E-39],
"MOL_99":[239237.482,32082.19592,54.75337105,1,7.90E-39],
"MOL_118":[279444.233,37125.02875,68.01290084,1,1.36E-38],
"MOL_225":[539587.736,69384.25858,72.11619521,1,3.01E-38]}

# Calculate the translational, rotational and total entropy respectively
assembly_results = []
for key, value in solute_property.items():
    S_t, S_r, S = calculate_entropy(solvent_property, value)
    assembly_results.append({
        "Key": key,
        "SBU": SBU[len(assembly_results)],  
        "BTB": BTB[len(assembly_results)],  
        "S_t": S_t,
        "S_r": S_r,
        "S": S
    })
# Save the result to CSV
df = pd.DataFrame(assembly_results)
output_csv = "assembly_entropy.csv"
df.to_csv(output_csv, index=True)

# Visualize the result
plt.figure()
df_sbu=df.copy()
df_sbu = df_sbu.sort_values(by="SBU")
plt.plot(df_sbu["SBU"],df_sbu["S_t"],label="Translational Entropy",c="b")
plt.plot(df_sbu["SBU"],df_sbu["S_r"],label="Rotational Entropy",c="r")
plt.plot(df_sbu["SBU"],df_sbu["S"],label="Entropy",c="g")
plt.xlabel("The number of SBUs")
plt.ylabel("Entropy/R")
plt.legend(loc="upper right")

plt.figure()
df_btb=df.copy()
df_btb = df_btb.sort_values(by="BTB")
plt.plot(df_btb["BTB"],df_btb["S_t"],label="Translational Entropy",c="b")
plt.plot(df_btb["BTB"],df_btb["S_r"],label="Rotational Entropy",c="r")
plt.plot(df_btb["BTB"],df_btb["S"],label="Entropy",c="g")
plt.xlabel("The number of BTBs")
plt.ylabel("Entropy/R")
plt.legend(loc="upper right")
plt.show()

In [ ]:
# Apply first-order differencing to the data
sbu = df_sbu['SBU'].values
btb = df_btb['BTB'].values
entropy_sbu = df_sbu['S'].values
entropy_btb=df_btb['S'].values
# Generate an interpolation function (Entropy vs SBU)
interp_func_sbu = interp1d(sbu, entropy_sbu, kind='linear',fill_value="extrapolate")
# Generate an interpolation function (Entropy vs BTB)
interp_func_btb = interp1d(btb, entropy_btb, kind='linear',fill_value="extrapolate")

new_sbu = np.arange(1, 200, 1)
new_btb = np.arange(1, 350, 1)

# Calculate the interpolated entropy values
new_entropy_sbu = interp_func_sbu(new_sbu)
new_entropy_btb= interp_func_btb(new_btb)
plt.figure()
plt.plot(new_sbu,new_entropy_sbu,c="b")
plt.xlabel("The number of SBUs")
plt.ylabel("Entropy/R")
plt.title("The Interpolated Entropy vs SBU")
plt.figure()
plt.plot(new_btb,new_entropy_btb,c="r")
plt.title("The Interpolated Entropy vs BTB")
plt.xlabel("The number of BTBs")
plt.ylabel("Entropy/R")


# Calculte the first-order differencing of entropy vs SBU
entropy_sbu_diff = np.diff(new_entropy_sbu)
# Calculte the first-order differencing of entropy vs BTB
entropy_btb_diff  = np.diff(new_entropy_btb)
# Visualize the result
plt.figure()
plt.plot(new_sbu[1:],entropy_sbu_diff,c="b")
plt.xlabel("The number of SBUs")
plt.ylabel("Entropy/R")
plt.title("The First-Order Differencing of Entropy vs SBU")
plt.figure()
plt.plot(new_btb[1:],entropy_btb_diff,c="r")
plt.xlabel("The number of SBUs")
plt.title("The First-Order Differencing of Entropy vs BTB")
plt.ylabel("Entropy/R")
plt.show()
